In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [8]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

# large_model = init_chat_model("claude-sonnet-4-6")
# standard_model = init_chat_model("gpt-5-nano")
from langchain_openrouter import ChatOpenRouter
large_model = ChatOpenRouter(model="openai/gpt-5-nano") 
standard_model = ChatOpenRouter(model="openai/gpt-4.1-nano")   

@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I haven't watered the office plant today. Would you like me to take care of it now?


In [11]:
print(response["messages"][-1].response_metadata["model_name"])

openai/gpt-4.1-nano


In [12]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Typically when the plant becomes root-bound or the soil isn’t draining/holding moisture well anymore. For a normal office plant, that’s about every 1–2 years. Slower-growing plants might make it 2–3 years; faster growers may need it yearly.

Watch for these signs:
- Roots visibly circling the pot or coming out of drainage holes
- Soil drains unusually quickly or the plant stalls in growth
- Pots feel heavy and soil compacts even after watering

If you notice any of these, repot in spring into a pot 1–2 inches (2.5–5 cm) larger with fresh potting mix and good drainage. I can check the plant and give a precise recommendation if you want.


In [13]:
print(response["messages"][-1].response_metadata["model_name"])

openai/gpt-5-nano
